In [1]:
from pathlib import Path
import numpy as np
import re

In [2]:
def obtener_temp_maxima_filamentos(temperaturas, CF_ranges):
    """
    Obtiene la temperatura máxima para cada filamento.

    Parámetros:
    temperaturas (list o np.ndarray): Arreglo 1D con las temperaturas.
    CF_ranges (list): Lista de tuplas (inicio, fin) que delimitan cada filamento.

    Retorna:
    np.ndarray: Arreglo con la temperatura máxima de cada filamento.
    """
    temp_array = np.array(temperaturas)
    temperaturas_maximas = []

    for inicio, fin in CF_ranges:
        fragmento = temp_array[inicio : fin + 1]
        temp_max = np.max(fragmento)
        temperaturas_maximas.append(temp_max)

    return np.array(temperaturas_maximas)


In [3]:
def _extraer_fase_y_paso(nombre: str) -> tuple[str, int] | None:
    """
    Extrae fase y paso del nombre de un archivo Estado.

    Formato esperado: ``Estado_{fase}_sim_{N}_paso_{paso}.npz``

    Returns:
        (fase, paso) o None si el nombre no coincide.
    """
    m = re.match(r"Estado_(?P<fase>pp_set|sp_set|pp_reset|sp_reset)_sim_\d+_paso_(?P<paso>\d+)\.npz$", nombre)
    if m is None:
        return None
    return m.group("fase"), int(m.group("paso"))

In [4]:
def recopilar_archivos_fase(directorio: Path, fases_activas: list[str]) -> dict[str, list[tuple[int, Path]]]:
    """
    Escanea un directorio buscando archivos 'Estado_*.npz', extrae su fase y paso,
    y los devuelve agrupados por fase.
    """
    archivos_por_fase: dict[str, list[tuple[int, Path]]] = {}

    for fpath in directorio.glob("Estado_*.npz"):
        resultado = _extraer_fase_y_paso(fpath.name)
        if resultado is None:
            continue

        fase, paso = resultado
        if fase not in fases_activas:
            continue

        archivos_por_fase.setdefault(fase, []).append((paso, fpath))

    return archivos_por_fase


In [5]:
def obtener_ultimo_paso_con_temperatura(
    sim_path: Path,
    num_simulacion: int,
    fase: str = "pp_reset",
    subcarpeta: str = "reset",
) -> np.ndarray:
    """
    Busca en la subcarpeta de la simulación todos los archivos .npz de la fase
    indicada, los ordena por paso de mayor a menor, y devuelve la matriz de
    temperatura del ÚLTIMO paso donde ``"temperatura"`` sea un array
    (y no un escalar).

    Motivación: en pp_reset todos los pasos guardan la clave ``"temperatura"``,
    pero en los pasos finales (filamento ya rupturado) el campo se reduce a un
    escalar. Solo los pasos donde el filamento aún está activo contienen la
    distribución espacial completa (array 1-D con tantos elementos como nodos).

    Parámetros
    ----------
    sim_path : Path
        Ruta a la carpeta de la simulación (p. ej. ``Results/simulation_3``).
    num_simulacion : int
        Número de simulación (se usa solo para mensajes informativos).
    fase : str
        Fase a buscar. Por defecto ``"pp_reset"``.
    subcarpeta : str
        Subcarpeta dentro de sim_path donde están los archivos.
        Por defecto ``"reset"``.

    Returns
    -------
    np.ndarray
        Arreglo 1D de temperaturas del último paso válido.

    Raises
    ------
    FileNotFoundError
        Si no se encuentran archivos de la fase en la subcarpeta.
    ValueError
        Si en ningún archivo ``"temperatura"`` es un array (todos son escalares).
    """
    carpeta_fase = sim_path / subcarpeta

    if not carpeta_fase.is_dir():
        raise FileNotFoundError(f"Sim {num_simulacion}: no existe la subcarpeta '{carpeta_fase}'")

    # --- Recopilar y ordenar archivos de la fase solicitada ---
    patron = re.compile(rf"Estado_{re.escape(fase)}_sim_\d+_paso_(?P<paso>\d+)\.npz$")
    archivos: list[tuple[int, Path]] = []

    for fpath in carpeta_fase.glob(f"Estado_{fase}_*.npz"):
        m = patron.match(fpath.name)
        if m:
            archivos.append((int(m.group("paso")), fpath))

    if not archivos:
        raise FileNotFoundError(
            f"Sim {num_simulacion}: no se encontraron archivos de fase '{fase}' en '{carpeta_fase}'"
        )

    # Ordenamos de MAYOR a MENOR paso para iterar desde el más reciente
    archivos.sort(key=lambda x: x[0], reverse=True)

    # --- Buscar el último paso donde 'temperatura' sea un array (no escalar) ---
    for paso, fpath in archivos:
        with np.load(fpath) as datos:
            temp = datos["temperatura"]
            if temp.ndim >= 1 and temp.size > 1:
                print(
                    f"  Sim {num_simulacion}: temperatura matricial encontrada "
                    f"en paso {paso} (de {archivos[0][0]} pasos totales, "
                    f"tamaño={temp.size})"
                )
                return temp

    raise ValueError(
        f"Sim {num_simulacion}: 'temperatura' es escalar en todos los pasos de "
        f"la fase '{fase}' — no se puede extraer distribución por filamento."
    )

In [ ]:
carpeta_resultados = Path("Results")

# Rangos de nodos que pertenecen a cada filamento conductivo
CF_ranges = [(0, 59), (60, 119)]

datos_a_exportar = []

for sim_path in sorted(carpeta_resultados.glob("simulation_*")):
    num_sim = int(sim_path.name.split("_")[1])

    try:
        matriz_temperatura = obtener_ultimo_paso_con_temperatura(
            sim_path,
            num_sim,
            fase="pp_reset",
            subcarpeta="reset",
        )
        temp_maxima = np.round(obtener_temp_maxima_filamentos(matriz_temperatura, CF_ranges), 3)

        datos_a_exportar.append([num_sim, temp_maxima[0], temp_maxima[1]])
        print(f"  -> Temp. máx. por filamento: {temp_maxima}\n")

    except (FileNotFoundError, ValueError) as e:
        print(f"[AVISO] {e}\n")
        continue

# --- Exportar ---
if datos_a_exportar:
    matriz_final = np.array(datos_a_exportar)

    fich_name = "todas_temp_max_filamentos_pp_set.txt"
    np.savetxt(
        fich_name,
        matriz_final,
        fmt=["%d", "%.3f", "%.3f"],
        delimiter="\t",
        header="Simulacion\tTemp_Filamento_1 Set\tTemp_Filamento_2 Set",
        comments="",
    )
    print(f"¡Proceso finalizado! Datos guardados en '{fich_name}'")
else:
    print("No se encontraron datos de simulación para exportar.")


  Sim 1: temperatura matricial encontrada en paso 8400 (de 10000 pasos totales, tamaño=5040)
  -> Temp. máx. por filamento: [1330.054 1231.441]

  Sim 10: temperatura matricial encontrada en paso 7900 (de 10000 pasos totales, tamaño=5040)
  -> Temp. máx. por filamento: [ 845.228 1227.069]

  Sim 11: temperatura matricial encontrada en paso 8000 (de 10000 pasos totales, tamaño=5040)
  -> Temp. máx. por filamento: [848.202 756.126]

  Sim 12: temperatura matricial encontrada en paso 8100 (de 10000 pasos totales, tamaño=5040)
  -> Temp. máx. por filamento: [864.171 733.693]

  Sim 13: temperatura matricial encontrada en paso 8200 (de 10000 pasos totales, tamaño=5040)
  -> Temp. máx. por filamento: [1215.661  698.333]

  Sim 14: temperatura matricial encontrada en paso 8100 (de 10000 pasos totales, tamaño=5040)
  -> Temp. máx. por filamento: [ 862.113 1144.731]

  Sim 15: temperatura matricial encontrada en paso 8600 (de 10000 pasos totales, tamaño=5040)
  -> Temp. máx. por filamento: [ 85